# Бузя — резервное копирование и базовая настройка

Рабочая конфигурация проекта: `RWKV/RWKV7-G1l-1.5B-20261031`, архитектура RWKV-7.

Этот блокнот содержит две операции: резервное копирование локального snapshot модели на Google Drive и добавление технического сведения об архитектуре в системный промпт. Он не обучает и не изменяет веса модели.

**Важно:** сам открытый блокнот сохраняй через меню Colab (`Файл → Сохранить копию на Диске` или `Файл → Сохранить копию на GitHub`). Python runtime не предоставляет надёжного пути к открытому `.ipynb`.

Не загружай в публичный GitHub веса, приватную историю переписок, ключи API или личные данные.

In [ ]:
# 1. Подключаем Google Drive только если он ещё не подключён
import os
from google.colab import drive

if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Google Drive уже подключён.')

BASE_DIR = '/content/drive/MyDrive/Buzya'
os.makedirs(BASE_DIR, exist_ok=True)
print('Папка проекта:', BASE_DIR)

In [ ]:
# 2. Копируем полный snapshot выбранной модели RWKV-7 на Google Drive
# Если файлы уже есть в локальном HF-кэше, они возьмутся оттуда;
# если runtime был перезапущен и кэш пуст, Hugging Face скачает их заново.

import os
import shutil
from huggingface_hub import snapshot_download

MODEL_ID = 'RWKV/RWKV7-G1l-1.5B-20261031'
DEST = os.path.join(BASE_DIR, 'models', 'RWKV7-G1l-1.5B-20261031')
os.makedirs(os.path.dirname(DEST), exist_ok=True)

SOURCE = snapshot_download(repo_id=MODEL_ID)

total_bytes = sum(
    os.path.getsize(os.path.join(root, name))
    for root, _, files in os.walk(SOURCE)
    for name in files
)
print(f'Источник: {SOURCE}')
print(f'Размер snapshot: {total_bytes / (1024**3):.2f} GB')
print('Копирую файлы на Google Drive...')

shutil.copytree(SOURCE, DEST, dirs_exist_ok=True)

required_shards = [f'model-{i:05d}-of-00004.safetensors' for i in range(1, 5)]
missing = [name for name in required_shards if not os.path.isfile(os.path.join(DEST, name))]

if missing:
    raise RuntimeError('Не найдены части весов: ' + ', '.join(missing))

print('Копирование завершено. Все 4 части весов найдены.')
print('Папка:', DEST)
print('Файлов в snapshot:', sum(len(files) for _, _, files in os.walk(DEST)))

In [ ]:
# 3. Добавляем архитектуру RWKV-7 в system_prompt.txt без повторов

SYSTEM_PROMPT_PATH = os.path.join(BASE_DIR, 'memory', 'system_prompt.txt')
os.makedirs(os.path.dirname(SYSTEM_PROMPT_PATH), exist_ok=True)

architecture_note = (
    'Ты — языковая модель на архитектуре RWKV-7. '
    'Если тебя спрашивают, какая у тебя архитектура, отвечай: RWKV-7. '
    'Не называй свою архитектуру Transformer.'
)

if os.path.exists(SYSTEM_PROMPT_PATH):
    with open(SYSTEM_PROMPT_PATH, 'r', encoding='utf-8') as f:
        current_prompt = f.read()
else:
    current_prompt = ''

if architecture_note not in current_prompt:
    separator = '\n\n' if current_prompt.strip() else ''
    with open(SYSTEM_PROMPT_PATH, 'a', encoding='utf-8') as f:
        f.write(separator + architecture_note + '\n')
    print('Сведения об архитектуре добавлены.')
else:
    print('Сведения об архитектуре уже есть — повторно не добавляю.')

print('Системный промпт:', SYSTEM_PROMPT_PATH)

## Сохранение самого блокнота

Открытый блокнот Colab не обязательно доступен как файл внутри `/content`, поэтому не пытайся копировать его через `shutil` из runtime.

- В Google Drive: `Файл → Сохранить копию на Диске`.
- В GitHub: `Файл → Сохранить копию на GitHub`, затем выбери репозиторий `comedyfox6-oss/Buzya2` и папку `notebooks/`.

Эта копия содержит код и инструкции. Веса модели остаются в Google Drive, а не в публичном GitHub.